## 기본 체인

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

In [ ]:
from langchain_openai import ChatOpenAI

PROVIDER = "groq"

LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini", 
    },
}
MAX_RETRIES = 5  


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed, 
        max_retries=MAX_RETRIES,
    )

llm = get_llm(temperature=0.7) 

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = """당신은 집에 있는 재료로 쉽게 만들 수 있는 요리를 추천하는 친절한 요리사입니다.

[보유 재료]
{ingredients}

[지시사항]
- 위 재료를 주재료로 활용한 요리 1가지를 추천하세요.
- 소금, 설탕, 식용유 같은 기본 양념은 자유롭게 사용해도 됩니다.
- 조리법은 초보자도 따라 할 수 있도록 3~6단계로 작성하세요.

[출력 형식]
{ingredients}(으)로 만들 수 있는 요리를 추천드립니다!

추천 요리: (요리 이름)
재료: (사용 재료)
조리법:
1. ...
2. ...
"""

prompt = PromptTemplate.from_template(template)

print(prompt.format(ingredients="계란, 밥, 김치"))
print("=" * 50)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"ingredients": "계란, 밥, 김치"})
print(result)

In [ ]:
test_inputs = [
    {"ingredients": "토마토, 양파, 치즈"},
    {"ingredients": "두부, 애호박, 된장"},
    {"ingredients": "닭가슴살, 브로콜리, 마늘"},
]

results = chain.batch(test_inputs)

# zip으로 입력과 결과를 짝지어 출력
for inp, res in zip(test_inputs, results):
    print(f"입력 재료: {inp['ingredients']}")
    print(res)
    print("-" * 50)